# 02 · Snapshot features and targets

One account at a scoring date is the unit of prediction. Feature history is strictly earlier. Targets are half-open 90/180-day windows. Frequency means purchase days.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RUN = Path(__import__("os").environ.get("AMAZON_CLV_RUN", str(ROOT / "runs" / "main")))
assert (RUN / "manifest.json").exists(), "Run python -m amazon_clv run first"
audit = json.loads((RUN / "data_audit.json").read_text())
manifest = json.loads((RUN / "manifest.json").read_text())
pd.set_option("display.max_columns", 20)
print("Source MD5:", audit["md5"], "| Model version:", manifest["run_version"])


Source MD5: 3549d72735b1211ab009ff0177b94bd9 | Model version: amazon-clv-1.0-6efb098a6e


In [2]:
from amazon_clv.features import make_snapshot, feature_columns, training_rows
mapping = json.loads((RUN / "category_mapping.json").read_text())
categories = mapping["categories"][:-2]
snapshots = pd.read_parquet(RUN / "snapshots.parquet")
display(pd.DataFrame({"feature":mapping["feature_names"],"category":mapping["categories"]}))
print("Numeric feature count:",len(feature_columns(snapshots)))
display(snapshots.groupby("scoring_date").agg(accounts=("customer_id","size"),available_90=("target_90","count"),available_180=("target_180","count"),zeros_90=("target_90",lambda s:s.eq(0).sum())).tail(12))

,feature,category
0,category_share_00,ABIS_BOOK
1,category_share_01,GIFT_CARD
2,category_share_02,PET_FOOD
3,category_share_03,SHOES
4,category_share_04,NOTEBOOK_COMPUTER
5,category_share_05,NUTRITIONAL_SUPPLEMENT
6,category_share_06,HEADPHONES
7,category_share_07,PHYSICAL_VIDEO_GAME_SOFTWARE
8,category_share_08,SHIRT
9,category_share_09,HEALTH_PERSONAL_CARE


Numeric feature count: 51


,accounts,available_90,available_180,zeros_90
scoring_date,,,,
2021-11-01,4967,4967,4967,510
2021-12-01,4978,4978,4978,534
2022-01-01,4991,4991,4991,627
2022-02-01,4998,4998,4998,666
2022-03-01,4998,4998,4998,658
2022-04-01,5001,5001,5001,642
2022-05-01,5004,5004,5004,628
2022-06-01,5007,5007,0,633
2022-07-01,5008,5008,0,624


## Demonstrate future invariance
Append a future high-value transaction and recompute the earlier snapshot. Every predictive feature must remain exactly unchanged. Targets may change when future observations change, as they should.

In [3]:
tx = pd.read_parquet(RUN / "transactions.parquet")
ids = snapshots.loc[snapshots.scoring_date.eq("2022-01-01"),"customer_id"].head(12)
small = tx.loc[tx.customer_id.astype(str).isin(ids)].copy()
before = make_snapshot(small,"2022-01-01",categories)
extra = small.iloc[[0]].copy()
extra["date"] = pd.Timestamp("2022-02-01")
extra["value_cents"] = 99999999
after = make_snapshot(pd.concat([small,extra],ignore_index=True),"2022-01-01",categories)
pd.testing.assert_frame_equal(before[["customer_id"]+feature_columns(before)],after[["customer_id"]+feature_columns(after)])
print("PASS: adding future purchases cannot change earlier features")
display(before[["customer_id","recency_days","history_days","spend_90","purchase_days_90","target_90","target_180"]].head())

PASS: adding future purchases cannot change earlier features


,customer_id,recency_days,history_days,spend_90,purchase_days_90,target_90,target_180
0,R_01vNIayewjIIKMF,3,1124,798.52,5.0,279.44,417.18
1,R_037XK72IZBJyF69,4,1444,1635.25,20.0,1626.27,2152.84
2,R_038ZU6kfQ5f89fH,41,1110,184.31,2.0,333.54,333.54
3,R_03aEbghUILs9NxD,4,1449,248.18,13.0,160.55,203.63
4,R_06RZP9pS7kONINr,4,1439,1035.26,11.0,668.21,1178.28


## Enforce label availability
Historical features alone are insufficient. A training target must also finish before the scoring cutoff. The final fit can include mature 2021 examples after validation choices freeze.

In [4]:
split = pd.read_csv(RUN / "split_audit.csv")
assert (pd.to_datetime(split.max_label_end) < pd.to_datetime(split.cutoff)).all()
display(split)
for h in [90,180]:
    train = training_rows(snapshots,h,"2022-01-01")
    assert train.scoring_date.lt("2022-01-01").all()
    assert train[f"target_{h}"].notna().all()
    print(f"{h}d final training rows: {len(train):,}")

,stage,horizon,cutoff,train_n,max_train_scoring_date,max_label_end,validation_n,category_fit_end_exclusive
0,validation,90,2021-01-01,101010,2020-10-01,2020-12-30,4882.0,2019-01-01
1,validation,90,2021-07-01,110740,2020-12-01,2021-03-01,4932.0,2019-01-01
2,validation,180,2021-01-01,86523,2020-07-01,2020-12-28,4882.0,2019-01-01
3,validation,180,2021-07-01,110740,2020-12-01,2021-05-30,4932.0,2019-01-01
4,final_fit,90,2022-01-01,159960,2021-10-01,2021-12-30,NaN,2019-01-01
5,final_fit,180,2022-01-01,145118,2021-07-01,2021-12-28,NaN,2019-01-01


90d final training rows: 159,960
180d final training rows: 145,118


## Design tradeoffs
Freeze top categories from 2018 to eliminate future category popularity from the representation. UNKNOWN handles missing data and OTHER handles new categories. History length measures observed tenure, not true account age. Month sine/cosine features encode cyclic seasonality. Identifiers and late survey responses never enter features. Read DATA_DICTIONARY.md for every formula.